<a href="https://colab.research.google.com/github/SnehaVasishth/FineTune/blob/main/Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df=messages=pd.read_csv('/content/SMSSpamCollection.txt', sep='\t', names=['label','message'])

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.shape

(5572, 2)

In [ ]:
X=list(df['message'])

In [ ]:
y=list(df['label'])

In [ ]:
y=list(pd.get_dummies(y,drop_first=True)['spam'])

In [ ]:
y

[False,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 True,
 True,
 False,
 True,
 True,
 False,
 False,
 True,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 True,
 True,
 False,
 True,
 False,
 False,
 False,
 False

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=101)

In [ ]:
X_train

['Then we wait 4 u lor... No need 2 feel bad lar...',
 'Wat makes some people dearer is not just de happiness dat u feel when u meet them but de pain u feel when u miss dem!!!',
 "FR'NDSHIP is like a needle of a clock. Though V r in d same clock, V r nt able 2 met. Evn if V meet,itz only 4few seconds. Bt V alwys stay conected. Gud 9t;-)",
 'Its sarcasm.. .nt scarcasim',
 "Tell me they're female :V how're you throwing in? We're deciding what all to get now",
 'Leave it. U will always be ignorant.',
 'Aiyah e rain like quite big leh. If drizzling i can at least run home.',
 "Babe, I'm answering you, can't you see me ? Maybe you'd better reboot YM ... I got the photo ... It's great !",
 "K come to nordstrom when you're done",
 'What pa tell me.. I went to bath:-)',
 "Jus telling u dat i'll b leaving 4 shanghai on 21st instead so we'll haf more time 2 meet up cya...",
 'Ok..',
 '1) Go to write msg 2) Put on Dictionary mode 3)Cover the screen with hand, 4)Press  &lt;#&gt; . 5)Gently remove 

In [ ]:
!pip install transformers

In [ ]:
# Step 5: Tokenization using DistilBERT tokenizer
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Return TensorFlow tensors directly
train_enc = tokenizer(X_train, truncation=True, padding=True, return_tensors='tf')
test_enc = tokenizer(X_test, truncation=True, padding=True, return_tensors='tf')


In [ ]:
# Step 6: Convert labels to TensorFlow tensors
import tensorflow as tf

y_train_tensor = tf.convert_to_tensor(y_train, dtype=tf.int32)
y_test_tensor = tf.convert_to_tensor(y_test, dtype=tf.int32)


In [ ]:
# Step 7: Create TensorFlow datasets (inputs, labels)
train_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': train_enc['input_ids'],
        'attention_mask': train_enc['attention_mask']
    },
    y_train_tensor
)).shuffle(1000).batch(8)

test_dataset = tf.data.Dataset.from_tensor_slices((
    {
        'input_ids': test_enc['input_ids'],
        'attention_mask': test_enc['attention_mask']
    },
    y_test_tensor
)).batch(16)


In [ ]:
# Step 8: Load DistilBERT model for binary classification
from transformers import TFDistilBertForSequenceClassification

model = TFDistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2  # Binary classification
)


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_transform.bias', 'vocab_transform.weight', 'vocab_layer_norm.bias']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'classifier.bias']
You should 

In [ ]:
# Step 9: Create optimizer using Hugging Face utility
from transformers import create_optimizer

batch_size = 8
epochs = 2
num_train_steps = len(train_dataset) * epochs

optimizer, lr_schedule = create_optimizer(
    init_lr=2e-5,
    num_warmup_steps=100,
    num_train_steps=num_train_steps
)


In [ ]:
# Step 10: Compile model with correct loss function
model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)


In [53]:
# Step 11: Train the model!
model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=2
)


Epoch 1/2
558/558 [==============================] - 6597s 12s/step - loss: 0.1131 - accuracy: 0.9704 - val_loss: 0.0363 - val_accuracy: 0.9919
Epoch 2/2
558/558 [==============================] - 6593s 12s/step - loss: 0.0214 - accuracy: 0.9946 - val_loss: 0.0320 - val_accuracy: 0.9919


In [54]:
text = ["Congratulations! You won a free iPhone!"]
inputs = tokenizer(text, truncation=True, padding=True, return_tensors='tf')
output = model(inputs)
pred = tf.argmax(output.logits, axis=1).numpy()[0]
print("Spam" if pred == 1 else "Ham")


Spam
